In [0]:
# =============================================================
# writer.py
# Layer     : Bronze
# Purpose   : Delta writes only.
#             - Streaming write (fact tables via Auto Loader)
#             - Batch write   (dimension tables, full refresh)
#             - No business logic. No schema definitions.
#             - No column transformations.
#
# STRICT RULE: No readStream here. No StructType here.
#              No MERGE here — Silver owns merge logic.
# =============================================================


def write_stream(df, table_name: str, checkpoint_path: str) -> None:
    """
    Write a streaming DataFrame to Bronze Delta using append mode.

    Uses availableNow=True trigger — processes all currently available
    files in the Auto Loader queue and then stops. This gives us
    micro-batch incremental behaviour without a 24/7 running stream.

    Auto Loader checkpoint ensures files are never re-processed.

    Args:
        df              : Streaming DataFrame with lineage columns
        table_name      : e.g. "device_snapshots"
        checkpoint_path : abfss path for Auto Loader state
    """
    bronze_path = abfss("bronze", table_name)

    print(f"[writer] Stream → Bronze Delta : {bronze_path}")
    print(f"[writer] Checkpoint            : {checkpoint_path}")

    (
        df.writeStream
          .format("delta")
          .outputMode("append")
          .option("checkpointLocation",  checkpoint_path)
          .option("mergeSchema",         "true")       # allow schema evolution
          .trigger(availableNow=True)                  # process queue then stop
          .start(bronze_path)
          .awaitTermination()
    )

    print(f"[writer] Stream write complete : {table_name}")


def write_batch(df, table_name: str) -> None:
    """
    Write a batch DataFrame to Bronze Delta.

    Used for:
        - Dimension tables (always full overwrite)
        - Manual backfills

    overwriteSchema=True allows schema changes on full refresh.

    Args:
        df          : Batch DataFrame with lineage columns
        table_name  : e.g. "owners"
    """
    bronze_path = abfss("bronze", table_name)

    print(f"[writer] Batch → Bronze Delta : {bronze_path}")

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .save(bronze_path)
    )

    row_count = spark.read.format("delta").load(bronze_path).count()
    print(f"[writer] Batch write complete : {table_name}  ({row_count:,} rows)")


def get_bronze_row_count(table_name: str) -> int:
    """Quick row count from an existing Bronze Delta table."""
    bronze_path = abfss("bronze", table_name)
    return spark.read.format("delta").load(bronze_path).count()

print("[writer] Loaded. Bronze Delta writer ready.")